# Foucaultgram zone extraction

Extracts Foucault zone readings (zone radius vs. knife position) from a series of camera captures taken at different knife positions. The output JSON can be read directly by *Foucault test data reduction.ipynb*.

### Why not segment each image separately

The earlier exploratory notebook (*Shadowgram analysis.ipynb*) processed one image at a time: threshold at the median grey level, then find the light/shadow boundary. That approach is fragile for several reasons:

* The median threshold assumes light and shadow cover equal areas, which is generally false.
* Illumination is not uniform across the mirror (in this dataset the left side is much brighter), so a single global threshold shifts the detected boundary.
* Each image yields at most one zone, and Hough/Canny parameters need tuning per image.
* A knife that is slightly off the optical axis biases left and right zones in opposite directions. Measuring one side, or taking the boundary of a single image, does not cancel this bias.

### Stack approach

All images are treated as one stack $I(x, y, s)$, where $s$ is the knife position. Take a pupil point at horizontal position $x$ on the horizontal diameter (knife edge vertical). Its ray reaches the knife plane at a lateral distance of about $x\,(Z(x)-s)/R$ from the axis, where $Z(x)$ is the zone's normal intercept. The knife cuts a fraction of the source image, so

$$I(x, s) = I_\text{flat}(x)\; U\!\left(e - \frac{x\,(Z(x) - s)}{R}\right)$$

where $U$ is the knife-edge profile of the source image (going from 0 to 1) and $e$ is the lateral offset of the knife. Each pixel therefore goes through one dark/bright transition as the knife moves. The transition centre is at

$$s^*(x) = Z(x) - \frac{e\,R}{x}$$

Consequences:

* Each pixel is compared only with itself, so non-uniform illumination ($I_\text{flat}$) cancels out.
* Every pixel column along the diameter becomes a zone, giving hundreds of zones instead of one per image.
* Averaging left ($-x$) and right ($+x$) cancels the knife offset $e$, and the half-difference measures it: $\tfrac{1}{2}(s^*_R - s^*_L) = e R / r$.
* Transitions are wider near the centre (width $\propto R/|x|$), so inner zones are inherently less precise.

## 1. Configuration

In [ ]:
import json
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import OptimizeWarning, curve_fit, least_squares

plt.rcParams['figure.figsize'] = [9, 4]
plt.rcParams['figure.dpi'] = 110

DATASET_PATH = Path.cwd() / 'Datasets' / '(165) 13_06_2019' / '1'
OUTPUT_FILE = DATASET_PATH / 'zones_extracted.json'
CHANNEL = 1                   # OpenCV loads BGR; the light source is green
BAND_HALF_HEIGHT_PX = 20      # rows averaged above and below the horizontal diameter
X_SMOOTHING_PX = 2.0          # Gaussian smoothing along the diameter
MIN_CONTRAST = 30             # minimum bright level (counts) for a pixel to be used
R_BIN_MM = 2.0                # radial bin width of the exported zones
EDGE_MARGIN = 0.99            # ignore the outermost 1 % (edge diffraction ring)
SOURCE_MOVES_WITH_KNIFE = True  # matches "Source distance 0 = moving source" in Sixtests.set

## 2. Load the image stack

Only the green channel is used. Converting to grey would mix in the nearly empty red and blue channels, which add noise but no signal.

In [ ]:
data_base = json.loads((DATASET_PATH / 'mirror_measurements_database.json').read_text())
frames = sorted(data_base['measurements'], key=lambda m: m['optical_axis_offset'])
s = np.array([m['optical_axis_offset'] for m in frames])
stack = np.stack([cv2.imread(str(DATASET_PATH / m['foucaultgram']), cv2.IMREAD_UNCHANGED)[..., CHANNEL]
                  for m in frames]).astype(np.float32)

diameter = data_base['diameter']
R = data_base['radius_of_curvature_measurement']['radius_of_curvature']
a_mm = diameter / 2

print(f'{len(s)} frames, knife from {s[0]} to {s[-1]} mm, image {stack.shape[2]}x{stack.shape[1]} px')
print(f'saturated pixels (>= 254): {int((stack >= 254).sum())}, background level: {np.median(stack[:, :100, :100]):.1f} counts')

fig, axs = plt.subplots(1, 4, figsize=(14, 3))
for ax, k in zip(axs, np.linspace(0, len(s) - 1, 4).astype(int)):
    ax.imshow(stack[k], cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'knife {s[k]:.1f} mm'); ax.axis('off')
plt.show()

## 3. Locate the mirror

The per-pixel maximum over the stack shows the whole mirror lit, because every pixel is bright at one end of the knife scan. This makes the outline much easier to find than in any single shadowgram, where the shadowed parts merge with the background.

* An ellipse fitted to the outline checks the camera alignment (the axis ratio should be about 1).
* The scale along the measured diameter is taken from the left and right edges on that row, interpolated to sub-pixel precision.

In [ ]:
max_projection = cv2.GaussianBlur(stack.max(axis=0), (0, 0), 3)
edge_level = 0.1 * np.percentile(max_projection, 99.5)
mirror_mask = (max_projection > edge_level).astype(np.uint8)
contours, _ = cv2.findContours(mirror_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
(ellipse_cx, ellipse_cy), ellipse_axes, ellipse_angle = ellipse = cv2.fitEllipse(max(contours, key=cv2.contourArea))
row = int(round(ellipse_cy))


def edge_crossings(profile, level):
    """Sub-pixel positions where a row first and last rises above level."""
    i0, i1 = np.flatnonzero(profile > level)[[0, -1]]
    left = i0 - 1 + (level - profile[i0 - 1]) / (profile[i0] - profile[i0 - 1])
    right = i1 + (profile[i1] - level) / (profile[i1] - profile[i1 + 1])
    return left, right


x_left, x_right = edge_crossings(max_projection[row], edge_level)
x_center, a_px = (x_left + x_right) / 2, (x_right - x_left) / 2
mm_per_px = a_mm / a_px

print(f'ellipse centre ({ellipse_cx:.1f}, {ellipse_cy:.1f}) px, axes {ellipse_axes[0]:.1f} x {ellipse_axes[1]:.1f} px, '
      f'axis ratio {min(ellipse_axes) / max(ellipse_axes):.4f}')
print(f'diameter row {row}: centre x = {x_center:.1f} px, radius = {a_px:.1f} px, scale = {mm_per_px * 1000:.1f} µm/px')

overlay = cv2.cvtColor(np.clip(max_projection, 0, 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
cv2.ellipse(overlay, ellipse, (255, 0, 0), 3)
cv2.rectangle(overlay, (int(x_left), row - BAND_HALF_HEIGHT_PX), (int(x_right), row + BAND_HALF_HEIGHT_PX), (0, 128, 255), 3)
plt.figure(figsize=(7, 5))
plt.imshow(overlay); plt.title('Max projection, fitted outline and analysed band'); plt.axis('off')
plt.show()

## 4. Knife-scan map

The band around the horizontal diameter is averaged vertically, giving one intensity profile per knife position. Each column of the map below is one pupil point normalised to its own range. In this view the whole test is visible at once:

* the left side goes from bright to dark and the right side from dark to bright;
* the transition moves to larger knife positions further from the centre (the longitudinal aberration of the zones);
* horizontal streaks are frame-to-frame brightness changes (source or camera exposure).

In [ ]:
band = stack[:, row - BAND_HALF_HEIGHT_PX:row + BAND_HALF_HEIGHT_PX + 1, :].mean(axis=1)
band = gaussian_filter1d(band, X_SMOOTHING_PX, axis=1)
x_mm = (np.arange(band.shape[1]) - x_center) * mm_per_px
inside = np.abs(x_mm) < EDGE_MARGIN * a_mm

span = np.maximum(band.max(axis=0) - band.min(axis=0), 1e-6)
normalised = (band - band.min(axis=0)) / span
plt.figure(figsize=(9, 4.5))
plt.imshow(normalised[:, inside], aspect='auto', cmap='gray',
           extent=[x_mm[inside][0], x_mm[inside][-1], s[-1] + 0.05, s[0] - 0.05])
plt.title('Knife-scan map (each column normalised)')
plt.xlabel('Position along horizontal diameter [mm]'); plt.ylabel('Knife position [mm]')
plt.show()

## 5. Transition position of each pixel

Two estimators of the transition centre $s^*(x)$ are compared. Both take the dark level as 0: the background is about 0 counts, so a fully cut pixel is black.

* **Half-maximum crossing**: the knife position where the intensity crosses half of the pixel's maximum, by linear interpolation between frames. Simple, but it relies on two frames only. It is biased when the bright plateau is not reached within the scan or when the source image has a wide halo (the slow intensity decline before the sharp drop seen below).
* **Sigmoid fit**: $I = A / (1 + e^{-(s - s^*)/w})$ fitted to all frames, starting from the half-maximum estimate. It uses every frame, so it averages frame-to-frame flicker. Fits whose centre falls outside the scan or is poorly determined are rejected.

In [ ]:
def half_max_crossing(intensity, s, rising):
    if not rising:
        intensity, s = intensity[::-1], s[::-1]
    level = 0.5 * intensity.max()
    k = np.flatnonzero(intensity >= level)[0]
    if k == 0:
        return np.nan
    return s[k - 1] + (level - intensity[k - 1]) / (intensity[k] - intensity[k - 1]) * (s[k] - s[k - 1])


def sigmoid(s, amplitude, centre, width):
    return amplitude / (1 + np.exp(-(s - centre) / width))


def fit_sigmoid(intensity, s, start, rising):
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', (OptimizeWarning, RuntimeWarning))
            p, cov = curve_fit(sigmoid, s, intensity, p0=[intensity.max(), start, 0.2 if rising else -0.2], maxfev=2000)
    except RuntimeError:
        return np.full(3, np.nan)
    # "not <" also rejects a NaN/inf variance
    if not (s[0] < p[1] < s[-1]) or not np.sqrt(cov[1, 1]) < 0.05:
        return np.full(3, np.nan)
    return p


transitions = {'half-maximum': np.full(x_mm.size, np.nan), 'sigmoid': np.full(x_mm.size, np.nan)}
sigmoid_params = np.full((x_mm.size, 3), np.nan)
for side in (-1, 1):
    columns = np.flatnonzero(inside & (np.sign(x_mm) == side))
    # Direction is decided per side: near the centre single pixels are too flat to tell.
    rising = np.median([np.corrcoef(s, band[:, i])[0, 1] for i in columns]) > 0
    for i in columns:
        if band[:, i].max() < MIN_CONTRAST:
            continue
        start = half_max_crossing(band[:, i], s, rising)
        transitions['half-maximum'][i] = start
        if not np.isnan(start):
            sigmoid_params[i] = fit_sigmoid(band[:, i], s, start, rising)
            transitions['sigmoid'][i] = sigmoid_params[i, 1]

for name, values in transitions.items():
    print(f'{name}: transition found for {np.count_nonzero(~np.isnan(values))} of {np.count_nonzero(inside)} pixel columns')

fig, ax = plt.subplots()
s_fine = np.linspace(s[0], s[-1], 200)
for n, x_target in enumerate((-85, -60, -45, 50, 70, 88)):
    i = np.argmin(np.abs(x_mm - x_target))
    ax.plot(s, band[:, i], 'o', color=f'C{n}', ms=3, label=f'x = {x_mm[i]:.0f} mm')
    if not np.isnan(sigmoid_params[i, 0]):
        ax.plot(s_fine, sigmoid(s_fine, *sigmoid_params[i]), color=f'C{n}')
ax.set(title='Pixel intensity vs. knife position (lines: sigmoid fits)', xlabel='Knife position [mm]', ylabel='Counts')
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
plt.show()

## 6. Combine left and right, and compare estimators

Transition positions are binned by zone radius on each side and the median is taken in each bin. Left and right are then averaged.

* The half-difference between sides gives the lateral knife offset $e$. It should be roughly constant with radius; a trend points to a tilted knife or a mis-centred mirror image.
* If the dataset contains manual readings in a Sixtests file (`Sixtests.set`), they are overlaid for validation.
* *Smoothness* is the RMS scatter about a low-order even polynomial. It measures noise, not accuracy.

In [ ]:
def bin_by_radius(x_mm, values, side, r_edges):
    m = ~np.isnan(values) & (np.sign(x_mm) == side)
    r, v = np.abs(x_mm[m]), values[m]
    idx = np.digitize(r, r_edges) - 1
    return np.array([np.median(v[idx == j]) if np.count_nonzero(idx == j) >= 3 else np.nan
                     for j in range(len(r_edges) - 1)])


def combine_sides(x_mm, values, r_edges):
    left = bin_by_radius(x_mm, values, -1, r_edges)
    right = bin_by_radius(x_mm, values, 1, r_edges)
    r = (r_edges[:-1] + r_edges[1:]) / 2
    zones = pd.DataFrame({'mirror_radius': r, 'left': left, 'right': right,
                          'optical_axis_offset': (left + right) / 2,
                          'knife_offset_um': 1000 * (right - left) / 2 * r / R})
    return zones.dropna(ignore_index=True)


def best_fit_conic(r, readings):
    """Moving-source normal intercept model with free knife zero; same as the reduction notebook."""
    def model(k):
        return r**2 / (R * (1 + np.sqrt(1 - (1 + k) * r**2 / R**2))) + np.sqrt(R**2 - (1 + k) * r**2) - R
    sol = least_squares(lambda p: model(p[1]) + p[0] - readings, x0=[0.0, -1.0])
    return sol.x[1]


def read_sixtests(path):
    lines = path.read_text().splitlines()
    start = next(i for i, line in enumerate(lines) if line.split('\t')[:2] == ['y', 'X']) + 1
    rows = [line.split('\t')[:2] for line in lines[start:lines.index('*')]]
    return np.array([[float(v.replace(',', '.')) for v in row] for row in rows])


r_edges = np.arange(0, EDGE_MARGIN * a_mm, R_BIN_MM)
zones = {name: combine_sides(x_mm, values, r_edges) for name, values in transitions.items()}
sixtests_file = DATASET_PATH / 'Sixtests.set'
manual = read_sixtests(sixtests_file) if sixtests_file.exists() else None

# Compare all sources over the radius range where every estimator found zones.
r_lo = max(z.mirror_radius.min() for z in zones.values())
r_hi = min(z.mirror_radius.max() for z in zones.values())
sources = {name: (z.mirror_radius.to_numpy(), z.optical_axis_offset.to_numpy()) for name, z in zones.items()}
if manual is not None:
    sources['manual (Sixtests)'] = (manual[:, 0], manual[:, 1])

summary = {}
for name, (r, readings) in sources.items():
    common = (r >= r_lo) & (r <= r_hi)
    rc, sc = r[common], readings[common]
    even_fit = np.polyval(np.polyfit(rc**2, sc, 2), rc**2)
    summary[name] = {'zones found': len(r), 'r range [mm]': f'{r.min():.0f} – {r.max():.0f}',
                     f'smoothness {r_lo:.0f}–{r_hi:.0f} mm [mm]': np.std(sc - even_fit),
                     f'best-fit k {r_lo:.0f}–{r_hi:.0f} mm': best_fit_conic(rc, sc)}
    if name in zones:
        summary[name]['knife offset [µm]'] = zones[name].knife_offset_um.median()
display(pd.DataFrame(summary).T.round(4))

fig, axs = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, (name, z) in zip(axs, zones.items()):
    ax.plot(z.mirror_radius, z.left, 'C0.', label='left')
    ax.plot(z.mirror_radius, z.right, 'C1.', label='right')
    ax.plot(z.mirror_radius, z.optical_axis_offset, 'k-', label='average')
    if manual is not None:
        ax.plot(manual[:, 0], manual[:, 1], 'rs', ms=4, label='manual (Sixtests)')
    ax.set(title=name, xlabel='Zone radius [mm]'); ax.grid(alpha=0.3)
axs[0].set_ylabel('Knife position [mm]'); axs[0].legend(fontsize=8)
plt.show()

## 7. Export the zones

The chosen estimator's averaged zones are written in the format expected by *Foucault test data reduction.ipynb*. Point its `DATA_FILE` to the file below.

In [ ]:
EXPORT_ESTIMATOR = 'sigmoid'

export = {
    'mirror_id': data_base['mirror_id'],
    'date': data_base['date'],
    'description': f"{data_base['description']} - extracted from {len(s)} foucaultgrams ({EXPORT_ESTIMATOR})",
    'diameter': diameter,
    'clear_aperture_diameter': diameter,
    'central_obstruction_diameter': 0.0,
    'target_conic_constant': -1.0,
    'tester': {'source_moves_with_knife': SOURCE_MOVES_WITH_KNIFE},
    'radius_of_curvature_measurement': data_base['radius_of_curvature_measurement'],
    'measurements': [{'optical_axis_offset': round(float(z.optical_axis_offset), 4),
                      'mirror_radius': round(float(z.mirror_radius), 2),
                      'foucaultgram': None}
                     for z in zones[EXPORT_ESTIMATOR].itertuples()],
}
OUTPUT_FILE.write_text(json.dumps(export, indent=2))
print(f'{len(export["measurements"])} zones written to {OUTPUT_FILE}')

## 8. Recommendations for image acquisition

What limits the result in this dataset, and what to change on the tester:

* **Scan range.** Inner zones (|x| ≲ 45 mm) null before the first frame on the left side, and the outermost right zones only near the last frame. Start the scan earlier and end it later, so that every zone of interest completes its transition with about 0.3 mm of margin on both sides.
* **Step size.** Transitions are only 2–4 frames wide at 0.1 mm per step. Steps of 0.02–0.05 mm (cheap with the motorised stage) make the fits much better constrained.
* **Flat and dark frames.** Capture one frame with the knife fully withdrawn and one with it fully in. Normalising each pixel by them removes the need to guess the bright level, which is the main weakness of the half-maximum estimator.
* **Fixed exposure.** Lock exposure, gain and white balance, and keep the source current stable. Frame-to-frame flicker is visible as streaks in the knife-scan map. Raw or monochrome capture avoids debayering artefacts.
* **Actual knife positions.** Record the stage position read back from the encoder instead of the commanded value, so that backlash does not enter the readings.
* **Camera alignment.** Keep the camera on axis and focused on the mirror, so that the outline stays round and the mm/px scale is uniform.
* **Other diameters.** The knife edge only measures slopes perpendicular to it. Rotate the mirror (e.g. 0°, 45°, 90°) and repeat to detect astigmatism or zones that are not rotationally symmetric, and compare with Bath.